# Import thư viện và Load dữ liệu

In [ ]:
import pandas as pd
import ast # Thư viện giúp chuyển đổi chuỗi string thành list

# 1. Đọc file dữ liệu đã lấy từ TMDB
movies_df = pd.read_csv('../data/processed/movies_enriched.csv')

# 2. Đọc file tags của người dùng MovieLens
tags_df = pd.read_csv('../data/raw/ml-latest-small/tags.csv')

print(f"Số lượng phim: {len(movies_df)}")
print(f"Số lượng tags: {len(tags_df)}")

Số lượng phim: 9621
Số lượng tags: 3683


# Xử lý file Tags

In [3]:
# Điền khoảng trắng cho các tag bị rỗng (nếu có)
tags_df['tag'] = tags_df['tag'].fillna('')

# Chuyển toàn bộ tag về chữ thường để đồng nhất
tags_df['tag'] = tags_df['tag'].str.lower()

# Gộp tất cả các tag của cùng 1 movieId thành một chuỗi (string)
tags_grouped = tags_df.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

# Đổi tên cột cho dễ hiểu
tags_grouped.rename(columns={'tag': 'user_tags'}, inplace=True)

print("✅ Đã gộp Tags thành công. Dữ liệu mẫu:")
tags_grouped.head(3)

✅ Đã gộp Tags thành công. Dữ liệu mẫu:


,movieId,user_tags
0,1,pixar pixar fun
1,2,fantasy magic board game robin williams game
2,3,moldy old


# Xử lý cột dạng Danh sách (List) bị biến thành Chuỗi (String)

In [4]:
# Hàm chuyển đổi chuỗi dạng list thành chuỗi cách nhau bằng dấu cách
def clean_list_column(text):
    if pd.isna(text):
        return ""
    try:
        # Biến chuỗi "['A', 'B']" thành list ['A', 'B']
        lst = ast.literal_eval(text) 
        # Nối các phần tử bằng khoảng trắng và chuyển về chữ thường
        return " ".join(lst).lower()
    except:
        return ""

# Áp dụng cho các cột chứa danh sách
columns_to_clean = ['tmdb_genres', 'keywords', 'cast']
for col in columns_to_clean:
    movies_df[col] = movies_df[col].apply(clean_list_column)

# Xử lý cột Overview (chỉ cần đưa về chữ thường và điền rỗng nếu thiếu)
movies_df['overview'] = movies_df['overview'].fillna('').str.lower()

print("✅ Đã chuẩn hóa xong các cột nội dung.")

✅ Đã chuẩn hóa xong các cột nội dung.


# Cooking

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Tải bộ từ điển stop words và công cụ rút gọn từ (chỉ chạy lần đầu)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Khởi tạo
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def remove_noise(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Loại bỏ tất cả các ký tự không phải là chữ cái tiếng Anh (xóa số và dấu câu)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text.lower())
    
    # 2. Xóa các khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 3. Tách từ, xóa stop words và đưa từ về nguyên thể (Lemmatization)
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    return " ".join(cleaned_words)

# Nối bảng tags_grouped vào movies_df dựa trên movieId (Left Join)
final_df = pd.merge(movies_df, tags_grouped, on='movieId', how='left')

# Điền rỗng cho những phim không có ai gắn tag
final_df['user_tags'] = final_df['user_tags'].fillna('')

# Hàm trộn tất cả thông tin thành một "nồi súp" văn bản
def create_soup(row):
    # Bạn có thể nhân đôi (trọng số) một trường nào đó nếu muốn nó quan trọng hơn.
    # Ví dụ: Nhân đôi genres hoặc keywords để mô hình chú ý hơn.
    return row['overview'] + " " + row['tmdb_genres'] + " " + row['keywords'] + " " + row['cast'] + " " + row['user_tags']

# Tạo cột soup
final_df['soup'] = final_df.apply(create_soup, axis=1)

print("🧹 Đang dọn dẹp nhiễu trong nồi Soup...")
# Áp dụng hàm dọn dẹp cho toàn bộ cột soup
final_df['soup'] = final_df['soup'].apply(remove_noise)

print("✅ Đã xử lý nhiễu thành công! Dưới đây là Soup sau khi dọn dẹp:")
print("-" * 50)
print(final_df['soup'].iloc[0])
print("-" * 50)

🧹 Đang dọn dẹp nhiễu trong nồi Soup...
✅ Đã xử lý nhiễu thành công! Dưới đây là Soup sau khi dọn dẹp:
--------------------------------------------------
led woody andy toy live happily room andy birthday brings buzz lightyear onto scene afraid losing place andy heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside difference family comedy animation adventure rescue friendship mission jealousy villain bullying elementary school rivalry anthropomorphism friend computer animation buddy walkie talkie toy car boy next door new toy neighborhood toy come life resourcefulness toy pixar tom hank tim allen rickles jim varney wallace shawn pixar pixar fun
--------------------------------------------------


# Xuất dữ liệu sạch

In [6]:
# Chỉ giữ lại các cột quan trọng cho Backend và Thuật toán
# Lưu ý: Ta giữ lại title và poster_path để sau này Backend trả về cho Web hiển thị
columns_to_keep = ['movieId', 'tmdbId', 'title', 'poster_path', 'soup']
clean_movies_df = final_df[columns_to_keep].copy()

# Lưu thành file mới
output_path = '../data/processed/movies_cleaned_soup.csv'
clean_movies_df.to_csv(output_path, index=False, encoding='utf-8')

print(f"🎉 Hoàn tất! File dữ liệu sạch đã sẵn sàng tại: {output_path}")

🎉 Hoàn tất! File dữ liệu sạch đã sẵn sàng tại: ../data/processed/movies_cleaned_soup.csv
